Movies ETL Pipeline using PySpark and PostgreSQL
Import Libraries

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import col, to_date, split
from pyspark.sql.functions import col, split, explode
from pyspark.sql.functions import col, trim
from pyspark.sql.functions import col, to_date, trim
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.functions import trim, when, col
from pyspark.sql.functions import split
from pyspark.sql.functions import explode
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

Create Spark Session

In [57]:
spark = SparkSession.builder \
    .appName("TMDB_Project") \
    .config("spark.jars", r"C:\Users\96655\Desktop\postgresql-42.7.7.jar") \
    .getOrCreate()

Load Dataset

In [58]:
df_raw = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("encoding", "UTF-8") \
    .csv(r"tmdb_movies/movies_20260211_033623.csv")

Explore Dataset

In [59]:
df_raw.show(10, truncate=False)

+------+------------------------------+------------------------------+------------+------------+------+---------------+----------+----------+----------------+---------------------------------------------+
|id    |title                         |original_title                |release_date|release_year|rating|rating_category|vote_count|popularity|popularity_level|genres                                       |
+------+------------------------------+------------------------------+------------+------------+------+---------------+----------+----------+----------------+---------------------------------------------+
|98    |Gladiator                     |Gladiator                     |2000-05-04  |2000        |8.221 |Excellent      |20487     |21.0347   |Low             |Action, Drama, Adventure                     |
|111646|Biyaheng Langit               |Biyaheng Langit               |2000-10-25  |2000        |2.0   |Poor           |2         |17.0227   |Low             |Drama, Action, Romance

Data Cleaning

In [60]:
df_clean = df_raw \
    .withColumn(
        "release_date_parsed",
        F.to_date("release_date", "yyyy-MM-dd")
    ) \
    .withColumn(
        "release_year_parsed",
        F.year("release_date_parsed")
    )

In [61]:
df_clean.select(
    "title",
    "release_date",
    "release_date_parsed",
    "release_year_parsed"
).show(10, False)

+------------------------------+------------+-------------------+-------------------+
|title                         |release_date|release_date_parsed|release_year_parsed|
+------------------------------+------------+-------------------+-------------------+
|Gladiator                     |2000-05-04  |2000-05-04         |2000               |
|Biyaheng Langit               |2000-10-25  |2000-10-25         |2000               |
|Malena                        |2000-03-16  |2000-03-16         |2000               |
|The Emperor's New Groove      |2000-12-15  |2000-12-15         |2000               |
|Snatch                        |2000-09-01  |2000-09-01         |2000               |
|American Psycho               |2000-01-21  |2000-01-21         |2000               |
|How the Grinch Stole Christmas|2000-11-17  |2000-11-17         |2000               |
|Scary Movie                   |2000-07-07  |2000-07-07         |2000               |
|Cast Away                     |2000-12-22  |2000-12-2

In [62]:
df_clean.filter(
    F.col("release_date_parsed").isNull()
).select("release_date").distinct().show(20, False)

+------------+
|release_date|
+------------+
+------------+



In [63]:
df = df_raw.withColumn(
    "release_date_safe",
    F.to_date("release_date", "yyyy-MM-dd")
)

In [64]:
df_valid = df.filter(F.col("release_date_safe").isNotNull())

In [65]:
df_valid.show(20, truncate=False)

+------+------------------------------+------------------------------+------------+------------+------+---------------+----------+----------+----------------+---------------------------------------------+-----------------+
|id    |title                         |original_title                |release_date|release_year|rating|rating_category|vote_count|popularity|popularity_level|genres                                       |release_date_safe|
+------+------------------------------+------------------------------+------------+------------+------+---------------+----------+----------+----------------+---------------------------------------------+-----------------+
|98    |Gladiator                     |Gladiator                     |2000-05-04  |2000        |8.221 |Excellent      |20487     |21.0347   |Low             |Action, Drama, Adventure                     |2000-05-04       |
|111646|Biyaheng Langit               |Biyaheng Langit               |2000-10-25  |2000        |2.0   |Poor 

In [66]:
df_valid.filter(F.year("release_date_safe") == 2000).show(10, truncate=False)

+------+------------------------------+------------------------------+------------+------------+------+---------------+----------+----------+----------------+---------------------------------------------+-----------------+
|id    |title                         |original_title                |release_date|release_year|rating|rating_category|vote_count|popularity|popularity_level|genres                                       |release_date_safe|
+------+------------------------------+------------------------------+------------+------------+------+---------------+----------+----------+----------------+---------------------------------------------+-----------------+
|98    |Gladiator                     |Gladiator                     |2000-05-04  |2000        |8.221 |Excellent      |20487     |21.0347   |Low             |Action, Drama, Adventure                     |2000-05-04       |
|111646|Biyaheng Langit               |Biyaheng Langit               |2000-10-25  |2000        |2.0   |Poor 

In [67]:
df_valid.orderBy(F.col("release_date_safe").desc()).show(10, truncate=False)

+-------+------------------------+------------------------+------------+------------+------+---------------+----------+----------+----------------+--------------------------------+-----------------+
|id     |title                   |original_title          |release_date|release_year|rating|rating_category|vote_count|popularity|popularity_level|genres                          |release_date_safe|
+-------+------------------------+------------------------+------------+------------+------+---------------+----------+----------+----------------+--------------------------------+-----------------+
|1598432|El Rostro               |El Rostro               |2026-12-31  |2026        |0.0   |Poor           |0         |0.0858    |Low             |Horror, Fantasy, Thriller       |2026-12-31       |
|1598464|Down the Road Apiece    |Down the Road Apiece    |2026-12-31  |2026        |0.0   |Poor           |0         |0.2313    |Low             |Drama, Western                  |2026-12-31       |
|1185

In [68]:
df_valid = df_valid.withColumn("release_year", F.year("release_date_safe"))

In [69]:
df = df.select(
    "id",
    "title",
    "original_title",
    "release_date",
    "release_year",
    "rating",
    "rating_category",
    "vote_count",
    "popularity",
    "popularity_level",
    "genres"
)

Connect to PostgreSQL

In [71]:
db_url = "jdbc:postgresql://localhost:5432/tmdb_db"

db_properties = {
    "user": "postgres",
    "password": "Asma1424",   # ← غيرها لكلمة المرور عندك
    "driver": "org.postgresql.Driver"
}

Load Data into PostgreSQL

In [72]:
df_valid.write \
    .format("jdbc") \
    .option("url", db_url) \
    .option("dbtable", "movies") \
    .option("user", db_properties["user"]) \
    .option("password", db_properties["password"]) \
    .option("driver", db_properties["driver"]) \
    .mode("overwrite") \
    .save()

print(" Data Successfully Written to PostgreSQL")

 Data Successfully Written to PostgreSQL


Read Data from PostgreSQL

In [73]:
df_from_db = spark.read \
    .format("jdbc") \
    .option("url", db_url) \
    .option("dbtable", "movies") \
    .option("user", db_properties["user"]) \
    .option("password", db_properties["password"]) \
    .option("driver", db_properties["driver"]) \
    .load()

In [74]:
total_rows = df.count()

if total_rows == 0:
    raise Exception("❌ Data Validation Failed: No valid rows remaining")

print("=================================")
print("✅ DATA VALIDATION PASSED")
print("Total Valid Rows:", total_rows)
print("=================================")

df_valid = df

✅ DATA VALIDATION PASSED
Total Valid Rows: 107997


In [75]:
print("Data Successfully Read Back From PostgreSQL")
df_from_db.show(5)

✅ Data Successfully Read Back From PostgreSQL
+------+--------------------+--------------------+------------+------------+------+---------------+----------+----------+----------------+--------------------+-----------------+
|    id|               title|      original_title|release_date|release_year|rating|rating_category|vote_count|popularity|popularity_level|              genres|release_date_safe|
+------+--------------------+--------------------+------------+------------+------+---------------+----------+----------+----------------+--------------------+-----------------+
|    98|           Gladiator|           Gladiator|  2000-05-04|        2000| 8.221|      Excellent|     20487|   21.0347|             Low|Action, Drama, Ad...|       2000-05-04|
|111646|     Biyaheng Langit|     Biyaheng Langit|  2000-10-25|        2000|   2.0|           Poor|         2|   17.0227|             Low|Drama, Action, Ro...|       2000-10-25|
| 10867|              Malena|              Malèna|  2000-03-16| 

In [76]:
query = "(SELECT COUNT(*) AS total_rows FROM movies_cleaned) AS tmp"

df_count = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/tmdb_db") \
    .option("dbtable", query) \
    .option("user", "postgres") \
    .option("password", "Asma1424") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_count.show()

+----------+
|total_rows|
+----------+
|     10000|
+----------+



In [77]:
query = "(SELECT * FROM movies_cleaned LIMIT 5) AS tmp"

df_preview = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/tmdb_db") \
    .option("dbtable", query) \
    .option("user", "postgres") \
    .option("password", "Asma1424") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_preview.show()

+-------+--------------------+--------------------+------------+----------+------------+----------+--------------------+--------------------+
|     id|               title|      original_title|release_date|popularity|vote_average|vote_count|              genres|        genres_array|
+-------+--------------------+--------------------+------------+----------+------------+----------+--------------------+--------------------+
|1084242|          Zootopia 2|          Zootopia 2|  2025-11-26|  397.7357|       7.682|       522|Animation, Comedy...|{Animation,Comedy...|
| 785663|           Old Henry|           Old Henry|  2021-10-01|  377.2589|       7.273|       638|Western, Action, ...|{Western,Action,D...|
| 533533|          TRON: Ares|          TRON: Ares|  2025-10-08|  355.6789|       6.553|       676|Science Fiction, ...|{"Science Fiction...|
|1180831|             Troll 2|             Troll 2|  2025-11-30|   332.688|        6.79|       231|Action, Fantasy, ...|{Action,Fantasy,T...|
|12282

In [78]:
df_preview.printSchema()

root
 |-- id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- popularity: double (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- vote_count: integer (nullable = true)
 |-- genres: string (nullable = true)
 |-- genres_array: string (nullable = true)



In [80]:
postgres_jar = "/path/to/postgresql-42.6.0.jar"  # استبدل بالمسار عندك

# إنشاء SparkSession
spark = SparkSession.builder \
    .appName("PostgresSchemaCheck") \
    .config("spark.jars", postgres_jar) \
    .getOrCreate()

# إعدادات الاتصال
db_url = "jdbc:postgresql://localhost:5432/tmdb_db"
db_user = "postgres"
db_pass = "Asma1424"

In [81]:
query_tables = "(SELECT table_name FROM information_schema.tables WHERE table_schema = 'public') AS tbls"

df_tables = spark.read \
    .format("jdbc") \
    .option("url", db_url) \
    .option("dbtable", query_tables) \
    .option("user", db_user) \
    .option("password", db_pass) \
    .option("driver", "org.postgresql.Driver") \
    .load()

print("schema public:")
df_tables.show()

جميع الجداول في schema public:
+--------------+
|    table_name|
+--------------+
|  movie_genres|
|  movies_clean|
|movies_cleaned|
|        movies|
+--------------+



In [82]:
query_columns = """
(SELECT table_name, column_name, data_type
 FROM information_schema.columns
 WHERE table_schema = 'public') AS cols
"""

df_columns = spark.read \
    .format("jdbc") \
    .option("url", db_url) \
    .option("dbtable", query_columns) \
    .option("user", db_user) \
    .option("password", db_pass) \
    .option("driver", "org.postgresql.Driver") \
    .load()

print("الأعمدة لكل جدول في schema public:")
df_columns.show(100, truncate=False)  # يعرض أول 100 صف كامل

الأعمدة لكل جدول في schema public:
+--------------+-----------------+----------------+
|table_name    |column_name      |data_type       |
+--------------+-----------------+----------------+
|movies_clean  |release_date_date|date            |
|movies_cleaned|id               |integer         |
|movies_clean  |vote_average     |double precision|
|movies_clean  |vote_count       |integer         |
|movies        |release_year     |integer         |
|movies_cleaned|release_date     |date            |
|movies_cleaned|popularity       |double precision|
|movies_cleaned|vote_average     |double precision|
|movies_cleaned|vote_count       |integer         |
|movies_clean  |id               |integer         |
|movies_clean  |popularity       |double precision|
|movies        |release_date_safe|date            |
|movie_genres  |genre            |text            |
|movies_clean  |title            |text            |
|movies_clean  |original_title   |text            |
|movies_clean  |release_date 